In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors, DenseVector
import random
import time


spark = SparkSession.builder \
    .appName("ClusteringAssignment") \
    .master("local[8]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

sc = spark.sparkContext
sc.setCheckpointDir("/tmp/spark-checkpoints")





26/04/18 18:10:07 WARN Utils: Your hostname, devmachine resolves to a loopback address: 127.0.1.1; using 172.31.239.142 instead (on interface eth0)
26/04/18 18:10:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/18 18:10:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

# =============================================================================
# readVectorsSeq
# =============================================================================

def readVectorsSeq(filename):
    """
    Reads a CSV file of feature vectors.
    Returns RDD[DenseVector], one vector per line.
    """
    return sc.textFile(filename).map(
        lambda line: Vectors.dense([float(x) for x in line.strip().split(",")])
    )


# =============================================================================
# kcenter  -- Farthest First Traversal
# =============================================================================

def kcenter(P, k):
    first_center = P.takeSample(False, 1)[0]
    centers = [first_center]

    # FIX: c=first_center captures value at definition time (not reference)
    distRDD = P.map(lambda p, c=first_center: (p, p.squared_distance(c))).cache()

    for i in range(1, k):
        farthest_center = distRDD.max(key=lambda x: x[1])[0]
        centers.append(farthest_center)

        old_distRDD = distRDD
        # FIX: c=farthest_center captures current loop value by value
        distRDD = distRDD.map(
            lambda x, c=farthest_center: (x[0], min(x[1], x[0].squared_distance(c)))
        ).cache()

        # Truncate DAG lineage every 10 iterations
        if i % 10 == 0:
            distRDD.checkpoint()
            distRDD.count()  # Force materialisation before continuing

        old_distRDD.unpersist()

    distRDD.unpersist()
    return centers


# =============================================================================
# _weighted_sample  -- distributed D^2 sampling helper
# =============================================================================

def _weighted_sample(distRDD, threshold):
    """
    Distributed weighted sampling without collecting the full dataset.

    Uses mapPartitions to compute local prefix sums per partition.
    Only one summary tuple per partition is sent to the driver
    (num_partitions items instead of all |P| items).

    This replaces the original distRDD.collect() inside the kmeansPP loop
    which sent ALL points to the driver on every single iteration.
    """
    def partition_prefix_sums(iterator):
        items = list(iterator)
        running = 0.0
        result = []
        for (p, d) in items:
            running += d
            result.append((running, p))
        yield (running, result)  # One tuple per partition

    # Collect only partition-level summaries (tiny data transfer)
    partition_summaries = distRDD.mapPartitions(partition_prefix_sums).collect()

    offset = 0.0
    for (part_total, part_items) in partition_summaries:
        for (local_cum, p) in part_items:
            if offset + local_cum >= threshold:
                return p
        offset += part_total

    # Fallback for floating-point edge cases
    for (_, part_items) in reversed(partition_summaries):
        if part_items:
            return part_items[-1][1]
    return None


# =============================================================================
# kmeansPP  -- k-means++ seeding (unweighted, for use on full dataset)
# =============================================================================

def kmeansPP(P, k):

    P = P.filter(lambda x: x is not None and isinstance(x, DenseVector)).cache()

    first_center = P.takeSample(False, 1)[0]
    if first_center is None:
        raise ValueError("Dataset is empty")
    centers = [first_center]

    distRDD = P.map(lambda p, c=first_center: (p, p.squared_distance(c))).cache()

    for i in range(1, k):
        total_dist = distRDD.map(lambda x: x[1]).sum()

        if total_dist == 0:
            new_center = distRDD.takeSample(False, 1)[0][0]
        else:
            r = random.random() * total_dist
            new_center = _weighted_sample(distRDD, r)

        if new_center is None:
            raise ValueError(f"Failed to select center at iteration {i}")

        centers.append(new_center)

        old_distRDD = distRDD
        distRDD = distRDD.map(
            lambda x, c=new_center: (x[0], min(x[1], x[0].squared_distance(c)))
        ).cache()

        if i % 5 == 0:
            distRDD.checkpoint()
            distRDD.count()

        old_distRDD.unpersist()

    distRDD.unpersist()
    return centers


# =============================================================================
# kmeansPP_weighted  -- weighted k-means++ for coreset (runs on driver)
# =============================================================================

def kmeansPP_weighted(X_weighted, k):

    if not X_weighted:
        raise ValueError("Empty weighted coreset")

    # Step 1: Pick first center weighted by cluster size
    total_weight = sum(w for (_, w) in X_weighted)
    r = random.random() * total_weight
    cumsum = 0.0
    first_center = X_weighted[-1][0]  # fallback
    for (p, w) in X_weighted:
        cumsum += w
        if cumsum >= r:
            first_center = p
            break

    centers = [first_center]

    # dists[i] = weight[i] * squared_distance(X_weighted[i][0], nearest_center)
    dists = [w * p.squared_distance(first_center) for (p, w) in X_weighted]

    for _ in range(1, k):
        total = sum(dists)

        if total == 0:
            # All coreset points coincide with existing centers
            new_center = max(range(len(X_weighted)),
                             key=lambda i: X_weighted[i][1])  # heaviest unassigned
            new_center = X_weighted[new_center][0]
        else:
            r = random.random() * total
            cumsum = 0.0
            new_center = X_weighted[-1][0]  # fallback
            for idx, (p, w) in enumerate(X_weighted):
                cumsum += dists[idx]
                if cumsum >= r:
                    new_center = p
                    break

        centers.append(new_center)

        # Update: dists[i] = min(old, weight[i] * D^2(point[i], new_center))
        for idx, (p, w) in enumerate(X_weighted):
            dists[idx] = min(dists[idx], w * p.squared_distance(new_center))

    return centers


# =============================================================================
# kmeansObj  -- k-means objective function
# =============================================================================

def kmeansObj(P, C):
    """
    Average squared distance of each point in P to its nearest center in C.

    Uses sc.broadcast(C) so the center list is serialised ONCE and cached
    on each executor, rather than being re-serialised for every task.

    Returns float (lower is better).
    """
    C_broadcast = sc.broadcast(C)

    def min_dist(p):
        return min(p.squared_distance(c) for c in C_broadcast.value)

    avg_obj = P.map(min_dist).mean()
    C_broadcast.unpersist()
    return avg_obj


# =============================================================================
# compute_coreset_weights
# =============================================================================

def compute_coreset_weights(P, coreset_centers):
    """
    Assigns each point in P to its nearest coreset center and counts
    how many points belong to each center (the weight).

    Args:
        P (RDD[DenseVector]): Original full dataset.
        coreset_centers (list[DenseVector]): Output of kcenter(P, k1).

    Returns:
        list of (DenseVector, int): Each coreset center with its count.
    """
    C_broadcast = sc.broadcast(coreset_centers)

    def nearest_center_idx(p):
        dists = [p.squared_distance(c) for c in C_broadcast.value]
        return min(range(len(dists)), key=lambda i: dists[i])

    # Count how many original points are assigned to each coreset center
    counts = (
        P.map(lambda p: (nearest_center_idx(p), 1))
         .reduceByKey(lambda a, b: a + b)
         .collect()
    )
    C_broadcast.unpersist()

    weight_map = dict(counts)
    # Centers with 0 points assigned (shouldn't happen but safe fallback)
    return [(coreset_centers[i], weight_map.get(i, 1))
            for i in range(len(coreset_centers))]


# =============================================================================
# run_experiment
# =============================================================================

def run_experiment(filename, k, k1):
    """
    Runs the three required experiments:

    1. kcenter(P, k)                              -> print running time
    2. kmeansPP(P, k) -> kmeansObj(P, C)          -> print objective
    3. kcenter(P,k1) -> weighted kmeansPP(X,k)
       -> kmeansObj(P, C)                         -> print objective
    """
    print("Reading dataset...")
    P = readVectorsSeq(filename)

    # Clean and cache once (avoids re-reading the file multiple times)
    P = P.filter(lambda x: x is not None and isinstance(x, DenseVector))
    dim = P.first().size
    P = P.filter(lambda x: x.size == dim).cache()

    n = P.count()
    print(f"Dataset loaded: {n} points, {dim} dimensions")

    # ------------------------------------------------------------------
    # Experiment 1: kcenter(P, k) -- time it
    # ------------------------------------------------------------------
    print(f"\n--- Experiment 1: kcenter(P, k={k}) ---")
    start = time.time()
    C_kcenter = kcenter(P, k)
    elapsed = time.time() - start
    print(f"Running time : {elapsed:.4f} sec")
    print(f"Centers found: {len(C_kcenter)}")

    # ------------------------------------------------------------------
    # Experiment 2: kmeansPP(P, k) -> kmeansObj
    # ------------------------------------------------------------------
    print(f"\n--- Experiment 2: kmeansPP(P, k={k}) ---")
    start = time.time()
    C_kmeans = kmeansPP(P, k)
    elapsed = time.time() - start
    print(f"Running time        : {elapsed:.4f} sec")
    obj1 = kmeansObj(P, C_kmeans)
    print(f"kmeansObj(P, C)     : {obj1:.6f}")

    # ------------------------------------------------------------------
    # Experiment 3: Weighted Coreset experiment
    #
    # Step 1: kcenter(P, k1) -> k1 spread-out coreset centers X
    # Step 2: compute_coreset_weights(P, X) -> (center, weight) pairs
    #         weight = number of original points nearest to that center
    # Step 3: kmeansPP_weighted(X_weighted, k) -> k final centers
    #         sampling prob proportional to weight * D^2
    # Step 4: kmeansObj(P, C) evaluated on the full original dataset
    # ------------------------------------------------------------------
    print(f"\n--- Experiment 3: Weighted Coreset "
          f"kcenter(P,k1={k1}) -> kmeansPP(X,k={k}) ---")

    start = time.time()
    X = kcenter(P, k1)
    elapsed_kc = time.time() - start
    print(f"kcenter(P, k1) time      : {elapsed_kc:.4f} sec")
    print(f"Coreset size             : {len(X)}")

    # Compute weights: how many original points each coreset center owns
    print("Computing coreset weights...")
    start = time.time()
    X_weighted = compute_coreset_weights(P, X)
    elapsed_w = time.time() - start
    print(f"Weight computation time  : {elapsed_w:.4f} sec")

    weights = [w for (_, w) in X_weighted]
    print(f"Weight stats - min: {min(weights)}, max: {max(weights)}, "
          f"mean: {sum(weights)/len(weights):.1f}, total: {sum(weights)}")

    # Weighted kmeans++ on coreset (small list, runs on driver)
    start = time.time()
    C_final = kmeansPP_weighted(X_weighted, k)
    elapsed_pp = time.time() - start
    print(f"kmeansPP_weighted time   : {elapsed_pp:.6f} sec")

    obj2 = kmeansObj(P, C_final)
    print(f"kmeansObj(P, C_coreset)  : {obj2:.6f}")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    print("\n========== Summary ==========")
    print(f"kmeansPP(P, k) objective      : {obj1:.6f}")
    print(f"Weighted coreset objective    : {obj2:.6f}")
    ratio = obj2 / obj1 if obj1 > 0 else float('inf')
    print(f"Ratio (coreset / direct)      : {ratio:.3f}x")
    print("(Ratio should be close to 1.0 -- ideally < 2.0)")

    P.unpersist()

In [3]:
if __name__ == "__main__":
    filename = "hdfs://localhost:9000/user/rajesh/spam_data/spambase.data"

    k  = 10   # Number of final clusters
    k1 = 100  # Coreset size (k1 >> k)

    run_experiment(filename, k, k1)

Reading dataset...


Dataset loaded: 4601 points, 58 dimensions

--- Experiment 1: kcenter(P, k=10) ---


Running time : 7.8928 sec
Centers found: 10

--- Experiment 2: kmeansPP(P, k=10) ---


Running time        : 9.3750 sec
kmeansObj(P, C)     : 26937.533586

--- Experiment 3: Weighted Coreset kcenter(P,k1=100) -> kmeansPP(X,k=10) ---


kcenter(P, k1) time      : 47.4054 sec
Coreset size             : 100
Computing coreset weights...


[Stage 155:>                                                        (0 + 2) / 2]

Weight computation time  : 0.9408 sec
Weight stats - min: 1, max: 2393, mean: 46.0, total: 4601
kmeansPP_weighted time   : 0.004106 sec
kmeansObj(P, C_coreset)  : 30149.621644

========== Summary ==========
kmeansPP(P, k) objective      : 26937.533586
Weighted coreset objective    : 30149.621644
Ratio (coreset / direct)      : 1.119x
(Ratio should be close to 1.0 -- ideally < 2.0)


In [4]:
spark.stop()